## 1. Introduction

Formula One (F1) has committed to achieving net-zero carbon emissions by 2030, yet a major share of its footprint still comes from the extensive global travel required each season. The current race calendar forces teams to repeatedly fly and ship equipment across continents, creating unnecessary logistical emissions. This project proposes a data-driven approach to redesigning the F1 calendar with sustainability in mind, using geospatial, climate, and cultural datasets to ensure any optimized schedule remains both practical and operationally realistic.


### 1.1 Problem Statement

The goal is to formulate the F1 race schedule as a routing optimization problem that minimizes total logistics-related carbon emissions. Each race venue is defined by its location, feasible weather window, and travel cost determined by distance and transport mode. 

The optimized calendar must respect real-world constraints such as fixed opening and closing races, atmost 3 consecutive events, and a mid-season break—while remaining operationally feasible. By solving this constrained optimization problem, the project aims to produce a race calendar that has the minimum CO2 emissions.


## 2. Approach

The model is built using sets of circuits and weekends, with binary decision variables that determine both the race order and the specific weekend assigned to each event. Additional variables capture positional ordering (via Miller–Tucker–Zemlin constraints), triple-header occurrences, long gaps between races, and the boundary around the summer break.

The constraints enforce the Traveling Salesman Problem (TSP) structure, wherein each circuit has exactly one predecessor and one successor, and each race is scheduled exactly once. We also ensure at most one race per weekend and eliminate subtours. Feasibility rules prevent races from being placed on weekends restricted by climate or cultural conditions. Temporal constraints link the race sequence with actual weekend assignments.

The model also embeds operational requirements: exactly 14 races before the summer break, races immediately before and after the break, and no races during the break period. Additional constraints control the number of triple-headers and limit excessively long gaps between races. Break-arc detection variables identify the transition cut by the break to correctly adjust emissions in the objective function. Overall, this formulation produces a season-long schedule that satisfies all logistical, climatic, and operational constraints while minimizing total freight-related emissions.

### 2.1 Modeling the summer break

We modelled the summer break by enforcing the below constraints:

- Exactly 14 races must be scheduled before the F1 summer break.
- The summer break spans (2026-08-09 to 2026-08-30) and no races can occur during this interval.
- A race must be scheduled on the weekends immediately before and after the summer break.

The summer break arc is the edge in the race graph that connects the last race before the summer break to the first race after the break. In reality, this arc does not exist, as teams travel back to their respective headquarters during the break. We use constraints on z[i, j] to identify this break arc. To model this correctly in the objective function, we subtract the emissions associated with the break arc, and instead add the emissions from pre-break race to HQ and the emission from HQ to post-break race.

### 2.2 Objective Function.

The total emission is calculated by taking the sum of the following items:

- **Start-of-season travel**: Emissions for all teams traveling from their respective HQ to the first race location.
- **Regular season travel**: Emissions for consecutive race-to-race travel, multiplied by 10 as there are 10 teams. (x[i, j] = 1 indicates race i followed by race j).
- **End-of-season travel**: Emissions for teams traveling from the final race back to their respective HQ.
- **Summer break emissions**:
    - Subtract emissions of the break arc.
    - Add emissions for pre-break race -> HQ and HQ -> post-break race travel.

### 2.2 Assumptions

#### 2.2.1 Representation of Race Locations

Each race location is represented by the geographical coordinates of its host city, serving as a proxy for the nearest international airport. This captures the primary logistics hubs used by Formula 1 teams without explicitly modeling short local transfers.

#### 2.2.2 Emission Factors and Freight Load

- Each team transports a standardized freight load of 50 tonnes.
- Emissions are calculated using tonne-kilometer factors:
    - Air: 0.13516 kg CO₂e/tonne-km
	- Road: 0.0165 kg CO₂e/tonne-km

#### 2.2.3 Modes of Transport

Only two logistics modes are modeled: air (for intercontinental/non-European transfers) and road (for intra-European transfers).

Furthermore, we made the following assumptions in our model:

- The Haversine distance is used to approximate flight emissions, assuming symmetric travel paths.
- A constant emission factor (kgCO₂e per km) is applied, with air-freight considered the dominant contributor.
- The opening and closing races are fixed as Australia and Abu Dhabi, respectively.
- Weekends with climate or cultural infeasibility are strictly disallowed.
- Triple-header limits and maximum gap constraints must be respected to account for real-world scheduling fatigue.
- The analysis includes only CO₂ emissions from freight transport between race locations and doesn't include other green house gases.

### 2.3 Data Collection & Preprocessing

#### 2.3.1 Circuit & Team Locations

- Circuit coordinates are stored in `circuits.csv`.
- Team headquarters are stored in `hq.csv`.
- Haversine distances between circuits, and between a circuit and HQ, are computed and stored in `distances.csv` .
- Climate data are stored in `climate.csv`
- Festival data are stored in `festival.csv`. These dates are marked infeasible in the feasibility matrix.

#### 2.3.2 Weather Feasibility

- For each circuit, five years of historical daily weather data were collected using the
Visual Crossing API.
- For each Sunday of 2026, the average temperature and precipitation across the past 5
years were computed.
- A weekend is marked feasible (1) if:
    - 10°C ≤ temperature ≤ 35°C
    - precipitation < threshold


## 3. GAMSPy Model

In [1]:
import sys
import numpy as np
import pandas as pd
import gamspy as gp

from src.parameters import *

gp.set_options({'USE_PY_VAR_NAME': 'yes'})
m = gp.Container()

In [2]:
# LOAD DATA
teams_df = pd.read_csv("data/processed/hq.csv")
circuits_df = pd.read_csv("data/processed/circuits.csv")
distances_df = pd.read_csv("data/processed/distances.csv")
climate_df = pd.read_csv("data/processed/climate.csv")
festivals_df = pd.read_csv("data/processed/festivals.csv")

In [3]:
rev_df = distances_df.rename(columns={"from": "to", "to": "from"})

distances_sym_df = (
    pd.concat([distances_df, rev_df], ignore_index=True)
      .drop_duplicates(subset=["from", "to"])
)

In [4]:
# CONSTANTS
BREAK_START_DATE = 24 # 2026-08-09 (in climate_df)
BREAK_END_DATE = 26   # 2026-08-30 (in climate_df) 
BREAK_POINT = 14      # NO. OF RACES BEFROE BREAK

# PARAMETERS identifying AUS and ABD
FIRST = "AUS"
LAST = "ABD"

# Identify last weekend before break and first after break
t_pre  = str(BREAK_START_DATE - 1)
t_post = str(BREAK_END_DATE   + 1)

MAX_TRIPLE_HEADERS = 3

In [5]:
# PREPROCESSING

teams = teams_df["team_id"].tolist()[:-1]
circuits = circuits_df["circuit_id"].tolist()

weekends = climate_df['week_num'].astype(int).tolist()
summer_break = climate_df.loc[
    climate_df['week_num'].between(BREAK_START_DATE, BREAK_END_DATE), 
    'week_num'].astype(int).tolist()


race_emissions = distances_sym_df[distances_sym_df['from'].isin(circuits) & 
                                  distances_sym_df['to'].isin(circuits)][["from", "to", "emissions_kgCO2e"]]
hq_emissions = distances_sym_df[distances_sym_df['from'].isin(teams)][["from", "to", "emissions_kgCO2e"]]

In [6]:
n_teams = len(teams)
n_races = len(circuits)
n_weekends = len(weekends)
team_ratio = 1

In [7]:
feasible_dates = np.zeros((n_weekends, n_races))
feasible_dates_df = pd.DataFrame(feasible_dates, columns=circuits, index=climate_df['week_num'].astype(int).tolist())

climate_df.index = climate_df['week_num'].astype(int).tolist()

for circuit in circuits:
    temp_col = f"{circuit}_avg_temp"
    precip_col = f"{circuit}_avg_precip"

    ok_temp = climate_df[temp_col].between(MIN_TEMP_F, MAX_TEMP_F)
    ok_precip = climate_df[precip_col] < MAX_PRECIP_IN

    feasible_dates_df.loc[ok_temp & ok_precip, circuit] = 1

# Festival filter
for idx, row in festivals_df.iterrows():
    feasible_dates_df.iloc[row['week_num'] -1 , feasible_dates_df.columns.get_loc(row['circuit_id'])] = 0

In [8]:
# Sentivity analysis:

# for i in circuits:
#     for j in circuits:
#         if i != j:
#             FIRST = i
#             LAST = j

In [9]:
# SET
Weekend = gp.Set(m, records=weekends)
SummerBreak = gp.Set(m, domain=[Weekend], records=summer_break)
Team = gp.Set(m, records=teams)
Circuit = gp.Set(m, records=circuits)
i = gp.Alias(m, alias_with=Circuit)
j = gp.Alias(m, alias_with=Circuit)
t = gp.Alias(m, alias_with=Weekend)



# PARAMETERS
r_emissions = gp.Parameter(m, domain=[Circuit, Circuit], records=race_emissions,
                         description="emissions between races in kgCO2e")
hq_emissions = gp.Parameter(m, domain=[Team, Circuit], records=hq_emissions,
                         description="emissions from HQ to race in kgCO2e")

feasible_dates = gp.Parameter(m, domain=[Weekend, Circuit], records=feasible_dates_df.stack().reset_index().values.tolist(),
                            description="feasibility of holding race on weekend (1 if feasible, 0 otherwise)")
feasible_dates[SummerBreak, Circuit] = 0 # No races during the break
    

first_set = gp.Parameter(m, records=14,
                         description='Number of races before the summer break')
break_start_date = gp.Parameter(m, records=np.array(BREAK_START_DATE),
                                description='Weekend where break begins')
break_end_date = gp.Parameter(m, records=np.array(BREAK_END_DATE),
                                description='Weekend where break ends')

In [10]:
# VARIABLES
x = gp.Variable(m,type='binary',domain=[Circuit,Circuit],
                description="1 if race in i is scheduled immediately before j, 0 otherwise")
y = gp.Variable(m, type='binary', domain=[Circuit,Weekend], 
                description='1 if race in Circuit is scheduled on Weekend, 0 otherwise')
z = gp.Variable(m, type="binary", domain=[i, j],
                description="1 if race i is on pre-break weekend and race j on post-break weekend")
u = gp.Variable(m,type='positive',domain=[Circuit], 
                description="Position of race in the calendar sequence")
gap = gp.Variable(m, type='binary', domain=[t],
                  description="1 if no race at week t")
triple_header = gp.Variable(m, type='binary', domain=[t],
                            description="1 if week t starts a triple header")


x.fx[i,i] = 0

u.lo[Circuit] = 2                   # all races but first must be at least position 2
u.up[Circuit] = gp.Card(Circuit)    # all races must be at most position number of races

u.fx[FIRST] = 1                # First race = AUS
u.fx[LAST] = gp.Card(Circuit)  # Last race = ABD (position = number of races)



In [ ]:
# EQUATIONS
assign1 = gp.Equation(m, domain=[j],
                      description='Each circuit has exactly one predecessor')
assign1[j] = gp.Sum(i, x[i,j]) == 1

assign2 = gp.Equation(m, domain=[i],
                      description='Each circuit has exactly one successor')
assign2[i] = gp.Sum(j, x[i,j]) == 1

assign3 = gp.Equation(m, domain=[i],
                      description='Each race is held exactly once')
assign3[i] = gp.Sum(t, y[i,t]) == 1

assign4 = gp.Equation(m, domain=[t],
                      description='Each weekend has at most one race')
assign4[t] = gp.Sum(i, y[i,t]) <= 1

# Get actual MTZ ordinal positions
FIRST_ord = circuits.index(FIRST) + 1
LAST_ord = circuits.index(LAST) + 1

# Then use ordinals instead of strings
mtz = gp.Equation(m, domain=[i,j])
mtz[i,j].where[(i.ord != FIRST_ord) & (j.ord != FIRST_ord) & (i.ord != LAST_ord)] = (
    u[i] - u[j] + 1 <= (gp.Card(Circuit) - 1) * (1 - x[i,j])
)

In [12]:
# Time Constraints
feasibiliy = gp.Equation(m, domain=[i,t],
                         description='Race can be held only if the weekend is feasible')
feasibiliy[i,t] = y[i,t] <= feasible_dates[t,i]


time_link = gp.Equation(m, domain=[i,j])
time_link[i,j].where[(i.ord != LAST_ord) | (j.ord != FIRST_ord)] = (
    gp.Sum(t, t.ord * y[j,t]) >= gp.Sum(t, t.ord * y[i,t]) + x[i,j] - (1 - x[i,j]) * gp.Card(Weekend)
)

# Triple Header Limits
triple_header_upper = gp.Equation(m, domain=[t],
                                  description='Direction 1: If triple_header=1, then must have 3 consecutive races')
triple_header_upper[t].where[t.ord <= gp.Card(Weekend) - 2] = (
    triple_header[t] * 3 <= gp.Sum(i, y[i, t]) + gp.Sum(i, y[i, t.lead(1)]) + gp.Sum(i, y[i, t.lead(2)])
)

triple_header_lower = gp.Equation(m, domain=[t],
                                  description='Direction 2: If there are 3 consecutive races, triple_header MUST be 1')
triple_header_lower[t].where[t.ord <= gp.Card(Weekend) - 2] = (
    gp.Sum(i, y[i, t]) + gp.Sum(i, y[i, t.lead(1)]) + gp.Sum(i, y[i, t.lead(2)]) <= 2 + triple_header[t]
)

triple_header_limit = gp.Equation(m,
                                  description='Limit on total triple headers')
triple_header_limit[...] = gp.Sum(t, triple_header[t]) <= MAX_TRIPLE_HEADERS



# Mid Season Break Condition
break_partition = gp.Equation(m,
                              description='Exactly 14 races before summer break')
break_partition[...] = gp.Sum([i, t], y[i,t].where[t.ord < BREAK_START_DATE]) == first_set

pre_break_race = gp.Equation(m,
                             description='Exactly one race on last weekend before break')
pre_break_race[...] = gp.Sum(i, y[i, str(BREAK_START_DATE - 1)]) == 1

post_break_race = gp.Equation(m,
                              description='Exactly one race on first weekend after break')
post_break_race[...] = gp.Sum(i, y[i, str(BREAK_END_DATE + 1)]) == 1


# Nice Gaps between races
gap_def = gp.Equation(m, domain=[t])
gap_def[t] = gap[t] == 1 - gp.Sum(i, y[i, t])

max_consecutive = gp.Equation(m, domain=[t])
max_consecutive[t].where[t.ord <= gp.Card(Weekend) - 3] = (
    gap[t] + gap[t.lead(1)] + gap[t.lead(2)] + gap[t.lead(3)] >= 1
)


In [13]:
z_lin1 = gp.Equation(m, domain=[i, j])
z_lin1[i, j] = z[i, j] <= x[i, j]

z_lin2 = gp.Equation(m, domain=[i, j])
z_lin2[i, j] = z[i, j] <= y[i, t_pre]

z_lin3 = gp.Equation(m, domain=[i, j])
z_lin3[i, j] = z[i, j] <= y[j, t_post]

z_lin4 = gp.Equation(m, domain=[i, j])
z_lin4[i, j] = z[i, j] >= x[i, j] + y[i, t_pre] + y[j, t_post] - 2


In [14]:
# Objective:
total_emissions = (
    # 1. HQ -> FIRST RACE
    gp.Sum(Team, hq_emissions[Team, FIRST]) * team_ratio
    
    # 2. Sum of all consecutive race emissions
    + gp.Sum([i, j], r_emissions[i, j] * x[i, j]) * 10
    
    # 3. Last race -> HQ
    + gp.Sum(Team, hq_emissions[Team, LAST]) * team_ratio

    # ---- SUMMER BREAK ADJUSTMENTS ----
    
    # 4. REMOVE emission of break connection
    - gp.Sum([i, j], r_emissions[i, j] * z[i, j]) * 10
    
    # 5. ADD: pre-break race -> HQ
    + gp.Sum([Team, i], hq_emissions[Team, i] * y[i, t_pre]) * team_ratio
    
    # 6. ADD: HQ -> post-break race
    + gp.Sum([Team, j], hq_emissions[Team, j] * y[j, t_post]) * team_ratio

    # ---- LOOP CLOSURE ADJUSTMENT ----
    
    # 7. REMOVE LAST RACE -> FIRST RACE
    - r_emissions[LAST, FIRST] * x[LAST, FIRST] * 10
)

model = gp.Model(
    m,
    name="f1_calendar",
    equations=m.getEquations(),
    sense=gp.Sense.MIN,
    problem=gp.Problem.MIP,
    objective=total_emissions
)

model.solve(solver="gurobi", options=gp.Options(
    time_limit=300,           # 5 minutes
    relative_optimality_gap=0.05
))

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,5.116557e+06,4644,2242,MIP,GUROBI,104.942


# 4. Results:

## 4.1 Visualization:

We visualize the travel between race circuits starting from the first race using plotly.

In [15]:
## Install plotly package
# !pip install plotly

In [16]:
import plotly.graph_objects as go

df = x.records                  # adjacency matrix in long form
df2 = u.records                 # each race with its order (u variable)
df1 = df[df['level'] == 1]      # edges actually used (x[i,j] = 1)

# Step 1: Race order
order_df = df2.sort_values("level").reset_index(drop=True)

# Step 2: Edges
edges = df1[['Circuit_0', 'Circuit_1']].reset_index(drop=True)

# Step 3: Coordinates
coords = circuits_df[['circuit_id', 'latitude', 'longitude']]

edges = (
    edges.merge(coords, left_on='Circuit_0', right_on='circuit_id')
         .rename(columns={'latitude':'lat_from', 'longitude':'lon_from'})
         .drop(columns=['circuit_id'])
         .merge(coords, left_on='Circuit_1', right_on='circuit_id')
         .rename(columns={'latitude':'lat_to', 'longitude':'lon_to'})
         .drop(columns=['circuit_id'])
)

fig = go.Figure()

# Function to create arrowhead
def arrowhead(lat_from, lon_from, lat_to, lon_to, scale=0.15):
    """Return a small arrowhead point slightly before the destination."""

    # Direction vector
    dlat = lat_to - lat_from
    dlon = lon_to - lon_from

    # Shorten vector for arrowhead base
    lat_arrow = lat_to - scale * dlat
    lon_arrow = lon_to - scale * dlon

    return lat_arrow, lon_arrow

# Draw route lines + arrowheads
for _, row in edges.iterrows():

    # Main route line
    fig.add_trace(go.Scattergeo(
        lon=[row['lon_from'], row['lon_to']],
        lat=[row['lat_from'], row['lat_to']],
        mode='lines',
        line=dict(width=2, color='red'),
        opacity=0.8,
        name=f"{row['Circuit_0']} → {row['Circuit_1']}"
    ))

    # Arrowhead segment
    lat_arrow, lon_arrow = arrowhead(row['lat_from'], row['lon_from'],
                                     row['lat_to'], row['lon_to'])

    fig.add_trace(go.Scattergeo(
        lon=[lon_arrow, row['lon_to']],
        lat=[lat_arrow, row['lat_to']],
        mode='lines',
        line=dict(width=4, color='red'),
        opacity=1.0,
        showlegend=False
    ))

# Circuit nodes
fig.add_trace(go.Scattergeo(
    lon=coords['longitude'],
    lat=coords['latitude'],
    mode='markers+text',
    marker=dict(size=8, color="blue"),
    text=coords['circuit_id'],
    textposition="top center",
    name="Circuits"
))

fig.update_layout(
    title="F1 Calendar Travel Route (with Direction Arrows)",
    geo=dict(
        projection_type="natural earth",
        showcountries=True,
        landcolor="rgb(240, 240, 240)",
    ),
    height=650,
    showlegend=False 
)

fig.show()


### 4.2 Sequence of circuit travel

The below table contains the sequence of circuit travel. All teams from their HQ location will travel to the first race (Australia) and then travel to Singapore for the 2nd race. From there on, they visit other circuits and have the final race at Abu Dhabi.

In [17]:
pd.merge(df1, df2, left_on = 'Circuit_0', right_on = 'Circuit').sort_values(by='level_y')[['Circuit_0', 'Circuit_1', 'level_y']]

,Circuit_0,Circuit_1,level_y
2,AUS,SIN,1.0
16,SIN,JPN,2.0
3,JPN,CHN,3.0
4,CHN,USA_LVG,4.0
19,USA_LVG,MEX,5.0
20,MEX,USA_COT,6.0
18,USA_COT,CAN,7.0
7,CAN,MIA,8.0
5,MIA,BRA,9.0
21,BRA,MAD,10.0


### 4.3 F1 Race Calendar

In [29]:
df = y.records
df = df[df['level'] == 1][["Circuit", "Weekend"]]

final_weekends = climate_df[['week_num', 'race_date']]
final_weekends = final_weekends.copy()
final_weekends['week_num'] = final_weekends['week_num'].astype('category')


df['Weekend'] = df['Weekend'].astype(int)
final_weekends['week_num'] = final_weekends['week_num'].astype(int)

final_weekends_1 = pd.merge(df, final_weekends,
                            left_on='Weekend',
                            right_on='week_num',
                            how='inner')
final_weekends_1 = final_weekends_1.sort_values(by='Weekend')
final_weekends_1 = final_weekends_1[['Circuit', 'race_date']]
final_weekends_1
# Merge with circuits dataframe to include country, circuit_name, and city
final_weekends_1 = pd.merge(final_weekends_1, circuits_df,
                            left_on='Circuit',
                            right_on='circuit_id',
                            how='inner')

# Select the final columns you want
final_weekends_1 = final_weekends_1[['circuit_name', 'city', 'country',  'race_date']]
final_weekends_1

,circuit_name,city,country,race_date
0,Albert Park Circuit,Melbourne,Australia,2026-03-01
1,Marina Bay Street Circuit,Singapore,Singapore,2026-03-08
2,Suzuka International Racing Course,Suzuka,Japan,2026-03-22
3,Shanghai International Circuit,Shanghai,China,2026-04-05
4,Las Vegas Strip Circuit,Las Vegas,USA,2026-04-12
5,Autódromo Hermanos Rodríguez,Mexico City,Mexico,2026-05-17
6,Circuit of the Americas,Austin,USA,2026-05-24
7,Circuit Gilles Villeneuve,Montreal,Canada,2026-06-07
8,Miami International Autodrome,Miami,USA,2026-06-14
9,Autódromo José Carlos Pace,São Paulo,Brazil,2026-06-28


The optimial minimum CO2 emission for the above F1 calendar is 5116557 kg CO₂e.